# 05. Biến thể 2: Pinball SVM (Pin-SVM)

## Cơ sở toán học: Sử dụng Pinball Loss để xử lý dữ liệu bất đối xứng.

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np, pandas as pd
from src.tools import load_features, tune_walk_forward, evaluate_on_test, summarize, train_test, scores
from src.models import PinSVM

data = load_features()

## Lưới tham số và chọn tham số

In [2]:
GRID = [
    {"kernel": ["linear"], "C": [0.01, 0.1, 1, 10], "tau": [0.1, 0.5, 1.0]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10], "tau": [0.1, 0.5, 1.0], "gamma": [0.01, 0.05, 0.2]},
]
best, rows = tune_walk_forward(PinSVM, GRID, data.loc[:"2022"])      # chạy khoảng 5-6 phút
json.dump({"best": best, "rows": rows}, open("../reports/tables/tuning_Pin-SVM.json", "w"), indent=1)
print("Tham số tốt nhất:", best)
pd.DataFrame([{**r["params"], "BalAcc_val": r["BalAcc"], "std": r["std"]} for r in rows]) \
  .sort_values("BalAcc_val", ascending=False).head(8)

Tham số tốt nhất: {'kernel': 'rbf', 'C': 0.1, 'tau': 0.5, 'gamma': 0.01}


,kernel,C,tau,BalAcc_val,std,gamma
15,rbf,0.1,0.5,0.524965,0.016785,0.01
12,rbf,0.1,0.1,0.524738,0.016861,0.01
18,rbf,0.1,1.0,0.523813,0.016399,0.01
27,rbf,1.0,1.0,0.522492,0.028061,0.01
24,rbf,1.0,0.5,0.520656,0.022590,0.01
21,rbf,1.0,0.1,0.519722,0.019988,0.01
16,rbf,0.1,0.5,0.513679,0.019767,0.05
13,rbf,0.1,0.1,0.512186,0.012176,0.05


## Đánh giá trên test

In [3]:
res = evaluate_on_test("Pin-SVM", PinSVM, best, data)
res.to_csv("../reports/tables/test_Pin-SVM.csv", index=False)
summarize(res, ["Accuracy", "BalAcc", "F1", "AUC", "MCC", "UpRate", "time_s", "nonzero"])

,Accuracy,BalAcc,F1,AUC,MCC,UpRate,time_s,nonzero
model,,,,,,,,
Pin-SVM,0.5328,0.5079,0.6227,0.5103,0.0165,0.6600,1.3208,1.0000


## Anh hưởng của τ, và phép thử τ = 0

In [4]:
X_train, y_train, X_test, y_test, *_ = train_test(data)
rows_tau = []
for tau in [0.0, 0.1, 0.3, 0.5, 1.0]:
    m = PinSVM(**{**best, "tau": tau}).fit(X_train, y_train)
    rows_tau.append({"tau": tau,
                     "nonzero": m.nonzero_ratio_,
                     "alpha<0 (điểm đúng bị phạt)": np.mean(m.lam_ < -1e-6 * best["C"]),
                     "BalAcc_test": scores(y_test, m.predict(X_test))["BalAcc"]})
tau_df = pd.DataFrame(rows_tau)
tau_df.to_csv("../reports/tables/pin_tau.csv", index=False)
tau_df

,tau,nonzero,alpha<0 (điểm đúng bị phạt),BalAcc_test
0,0.0,0.990757,0.000000,0.507639
1,0.1,1.000000,0.008666,0.504464
2,0.3,1.000000,0.008088,0.507937
3,0.5,1.000000,0.007510,0.507937
4,1.0,1.000000,0.005777,0.506349
